# 03 Outputs and Publishing

The `save_*()` functions write to the right `outputs/` folder and record each result in
`framework.db`. `publish()` sends results to the `local` folder destination in
`settings.yml`, which writes to `published/` (ignored by git).

In [ ]:
import json

import framework as fw
import matplotlib.pyplot as plt
import pandas as pd
import statsmodels.formula.api as smf

root = fw.scaffold()
bp = fw.data_read("inputs.final.bp_analysis")

## Tables, figures, models

A summary table of systolic and diastolic pressure by arm and visit.

In [ ]:
table = (
    bp.groupby(["treatment", "visit_number"])
    .agg(n=("systolic_bp", "size"), systolic=("systolic_bp", "mean"),
         diastolic=("diastolic_bp", "mean"))
    .round(1).reset_index()
)
fw.save_table(table, "bp_by_visit")

Jupyter shows the figure inline; `save_figure()` writes it to `outputs/figures/`.

In [ ]:
means = bp.groupby(["visit_number", "treatment"])["systolic_bp"].mean().unstack()
fig, ax = plt.subplots(figsize=(6, 4))
means.plot(ax=ax, marker="o", xticks=means.index,
           xlabel="Visit", ylabel="Mean systolic BP (mmHg)")
fw.save_figure(fig, "bp_trend", width=6, height=4)

An OLS model with the arm-by-visit interaction. `save_model()` pickles it.

In [ ]:
model = smf.ols("systolic_bp ~ visit_number * treatment + age + sex + bmi + smoker",
                data=bp).fit()
fw.save_model(model, "bp_ols")
model.params.round(2)

`save_notebook()` renders a notebook with Quarto and saves the HTML. Quarto doesn't
run an `.ipynb` by default, and these notebooks are kept without outputs, so
`--execute` re-runs `01-setup-and-data` first. Framework renders a copy, so the source
notebook stays without outputs. This cell takes a while.

In [ ]:
fw.save_notebook(root / "notebooks/01-setup-and-data.ipynb", name="setup-and-data",
                 quarto_args=["--execute"])

`save_report()` copies a finished file into `outputs/reports/`.

In [ ]:
draft = root / fw.settings("scratch") / "bp-summary.md"
effect = model.params["visit_number:treatment[T.intervention]"]
draft.write_text(f"# BP summary\n\nExtra change per visit with the intervention: "
                 f"{effect:.1f} mmHg.\n")
fw.save_report(draft, name="bp-summary")

`result_list()` lists everything saved, newest first.

In [ ]:
fw.result_list()[["name", "type", "created_at"]]

## Scratch

`scratch_capture()` writes an object to `scratch/` and returns it unchanged.
`scratch_clean()` empties the folder.

In [ ]:
fw.scratch_capture(bp.nlargest(10, "systolic_bp"), name="highest_bp.tsv")
print(sorted(p.name for p in (root / "scratch").iterdir()))
fw.scratch_clean()

## Publishing to a local folder

The `local` destination's relative `path` is resolved from the project root, so this
notebook writes to `<root>/published` even though it runs from `notebooks/`.

In [ ]:
fw.settings("publish")

A DataFrame becomes a table bundle, a figure an SVG bundle, and a model a structured
summary (coefficients and fit statistics only). Each returns the folder it wrote.

In [ ]:
fw.publish(table, slug="bp-by-visit", title="Blood pressure by visit")
fw.publish(fig, slug="bp-trend", title="Systolic BP trend")
fw.publish(model, slug="bp-ols", title="OLS model of systolic BP")

`publish_data()` publishes a DataFrame as a table, and `publish_dir()` publishes a folder
(here only the PNG figures). Both take a title. `publish_list()` lists the destination.

In [ ]:
fw.publish_data(table, "bp_by_visit_data.csv", title="BP by visit (data)")
fw.publish_dir(root / "outputs/figures", dest="bp-figures", pattern=r"\.png$",
               title="BP figures")
fw.publish_list()[["key", "last_modified"]]

`publish_notebook()` renders a notebook and publishes it. The title comes from the
notebook's first `# Heading` cell, "01 Setup and Data". It is rendered as saved, without
outputs.

In [ ]:
fw.publish_notebook(root / "notebooks/01-setup-and-data.ipynb", slug="setup-and-data")

Publishing a slug again keeps the old one under `v/<number>/`.

In [ ]:
fw.publish(table, slug="bp-by-visit", title="Blood pressure by visit")
sorted(p.name for p in (root / "published/bp-by-visit/v").iterdir())

Open `published/index.html` in a browser for the offline site. Its index:

In [ ]:
pd.DataFrame(json.loads((root / "published/index.json").read_text()))

## Cloud and buckets

The tour stays offline. `cloud_status()` only checks the local token store. Cloud and
bucket calls fail fast without a token or bucket, and nothing is sent.

In [ ]:
connected = fw.cloud_status()
if connected:
    fw.publish(table, to="cloud", slug="bp-by-visit")
for call in (fw.ledger_list, fw.storage_test, lambda: fw.publish_data(table, "t.csv", to="s3")):
    try:
        call()
    except (fw.CloudError, fw.FrameworkError) as error:
        print(str(error).splitlines()[0])